In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [6]:
data = pd.read_stata(r"Z:\survey\ECU\ENEMDU\1991\m11\data_orig\ECU_1991m11.dta") # para bases de stata
#data = pd.read_stata(r"datos/ECU_1991m11_BID.dta") # para bases de stata

## Revisar los datos

- rn - región natural
- estrato - estrato
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingobr - Ingresos como obrero o empleado
- ingpat - Ingresos como patrono o cuenta propia
- ingalq - Ingresos por alquileres, rentas o interese
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- fexp - factor de expansión
- ingrl - ingresos

El valor de 'ingobr' es el ingreso laboral monetario, no hay datos sobre ingreso laboral no monetario, las otras variables son ingreso no laboral monetario y no monetario e ingrl es un ingreso total

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no ocupado, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba ocupado tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39071 entries, 0 to 39070
Data columns (total 82 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   estrato   39071 non-null  int8    
 1   rn        39071 non-null  int8    
 2   ciudad    39071 non-null  object  
 3   zona      39071 non-null  object  
 4   sector    39071 non-null  object  
 5   vivienda  39071 non-null  object  
 6   hogar     39071 non-null  object  
 7   formul    39071 non-null  object  
 8   numpers   39071 non-null  int8    
 9   persona   39071 non-null  int8    
 10  resultad  39071 non-null  category
 11  reljefe   39071 non-null  category
 12  edad      39071 non-null  category
 13  sexo      39071 non-null  category
 14  nivinst   33810 non-null  category
 15  anoinst   31528 non-null  float64 
 16  asistea   32532 non-null  category
 17  sabele    29907 non-null  category
 18  iess      29907 non-null  category
 19  lininf    39071 non-null  object  
 20  donnac

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [9]:
data.columns

Index(['estrato', 'rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar',
       'formul', 'numpers', 'persona', 'resultad', 'reljefe', 'edad', 'sexo',
       'nivinst', 'anoinst', 'asistea', 'sabele', 'iess', 'lininf', 'donnac',
       'lugnac', 'siemvic', 'donvian', 'lugvian', 'cuanvic', 'trabajo',
       'actayuda', 'hortrasa', 'ratmeh', 'ratmeh1', 'ratmah', 'aunotra',
       'pornot', 'bustrasa', 'bustrama', 'amigos', 'directo', 'prensa',
       'agepu', 'agepri', 'tresne', 'tiembus', 'motnobus', 'deseatra',
       'condina', 'trabant', 'tiemnot', 'rama', 'grupo', 'catetrab',
       'pertrabn', 'numtrab', 'hortrahp', 'hortrahs', 'hortraho', 'ramas',
       'grupos', 'cates', 'ingobr', 'ingpat', 'ingalq', 'ingjub', 'ingotr',
       'oct', 'sep', 'ago', 'jul', 'jun', 'may', 'abr', 'mar', 'feb', 'ene',
       'dic', 'nov', 'condact', 'secins', 'subequ', 'ingrl', 'peamsiu',
       'fexp'],
      dtype='object')

In [10]:
data = data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'persona', 'numpers', 'edad', 'ingobr', 'ingpat', 'ingalq',
      'ingjub', 'ingotr', 'fexp', 'ingrl', 'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingobr' siempre que reportan estar ocupados en un mes

In [11]:
data['ingr_ene'] = data.apply(lambda x: x['ingobr'] if x['ene'] == 'ocupado' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingobr'] if x['feb'] == 'ocupado' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingobr'] if x['mar'] == 'ocupado' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingobr'] if x['abr'] == 'ocupado' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingobr'] if x['may'] == 'ocupado' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingobr'] if x['jun'] == 'ocupado' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingobr'] if x['jul'] == 'ocupado' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingobr'] if x['ago'] == 'ocupado' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingobr'] if x['sep'] == 'ocupado' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingobr'] if x['oct'] == 'ocupado' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingobr'] if x['nov'] == 'ocupado' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingobr'] if x['dic'] == 'ocupado' else None, axis=1)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\3865148962.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_ene'] = data.apply(lambda x: x['ingobr'] if x['ene'] == 'ocupado' else None, axis=1)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\3865148962.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_feb'] = data.apply(lambda x: x['ingobr'] if x['feb'] == 'ocupado' else None, axis=1)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\3865148962.py:3: SettingWithCopyWarning: 

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [12]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 1991]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc y tipo de cambio

In [13]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [14]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\2908473067.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ciudad'] = data['ciudad'].apply(str)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\2908473067.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\2908473067.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Tr

Diccionario ciudades disponibles

In [15]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\218289303.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))


### Asignamos el ipc y tipo de cambio correspondiente según trimestre y ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = \frac{ingr_{sucres}^{i}}{tipo-de-cambio^{i}}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [16]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [17]:
data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)
data['tipo_cambio_t1'] = tipo_cambio_dict.get(1)

data['ipc_t2'] = data.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data['ipc_base_t2'] = data.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)
data['tipo_cambio_t2'] = tipo_cambio_dict.get(2)

data['ipc_t3'] = data.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data['ipc_base_t3'] = data.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)
data['tipo_cambio_t3'] = tipo_cambio_dict.get(3)

data['ipc_t4'] = data.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data['ipc_base_t4'] = data.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)
data['tipo_cambio_t4'] = tipo_cambio_dict.get(4)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\2486551737.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\2486551737.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\2486551737.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of 

In [18]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\653647586.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\653647586.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\653647586.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [19]:
# Ingreso real por mes
data['ingr_ene_r'] = (data['ingr_ene'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_feb_r'] = (data['ingr_feb'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_mar_r'] = (data['ingr_mar'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_abr_r'] = (data['ingr_abr'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_may_r'] = (data['ingr_may'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jun_r'] = (data['ingr_jun'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jul_r'] = (data['ingr_jul'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_ago_r'] = (data['ingr_ago'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_sep_r'] = (data['ingr_sep'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_oct_r'] = (data['ingr_oct'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_nov_r'] = (data['ingr_nov'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_dic_r'] = (data['ingr_dic'] / data['tipo_cambio_t4']) * data['def_t4']

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1764603253.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_ene_r'] = (data['ingr_ene'] / data['tipo_cambio_t1']) * data['def_t1']
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1764603253.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_feb_r'] = (data['ingr_feb'] / data['tipo_cambio_t1']) * data['def_t1']
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1764603253.py:4: SettingWithCopyWarning: 
A value is trying to be set

Ingreso mensual promedio en el trimeste

In [20]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\210539431.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\210539431.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\210539431.py:3: SettingWithCopyWarning: 
A value is trying to be 

## Calculo ingreso de los hogares

In [21]:
columnas_idef = ['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\421910132.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)


8243

In [22]:
data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'idef_hogar', 'persona', 'numpers']]

,rn,estrato,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,1,3,010150,001,005,01,1,13010150001005011,1,3
1,1,3,010150,001,005,01,1,13010150001005011,2,3
2,1,3,010150,001,005,01,1,13010150001005011,3,3
3,1,3,010150,001,005,02,1,13010150001005021,1,2
4,1,3,010150,001,005,02,1,13010150001005021,2,2
...,...,...,...,...,...,...,...,...,...,...
39066,3,0,210450,001,011,11,1,30210450001011111,5,5
39067,3,0,210450,001,011,12,1,30210450001011121,1,4
39068,3,0,210450,001,011,12,1,30210450001011121,2,4
39069,3,0,210450,001,011,12,1,30210450001011121,3,4


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [23]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [24]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1616858308.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1616858308.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1616858308.py:3: SettingWithCopyWarning: 
A value is trying to be s

In [25]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h    9834.466361
ingr_t2_h     9034.19971
ingr_t3_h    8967.536506
ingr_t4_h    8067.372488
dtype: object

In [26]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  8067.372487502068
Mediana del ingreso de un hogar t4:  5696.641766448028


## Sacamos edades negativas y mayores a 100 años

In [27]:
len(data)

39071

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [28]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\3957095988.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)


In [29]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

39071

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [30]:
k = 0.4
s = 0.9

In [31]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [32]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [33]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [34]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...
39066,333.093127,305.323018,301.604505,272.655126
39067,1836.696621,1683.570481,1663.06636,1503.437652
39068,1836.696621,1683.570481,1663.06636,1503.437652
39069,1836.696621,1683.570481,1663.06636,1503.437652


In [35]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  2090.4652686046174
Mediana del ingreso individual descontando cargas familiares t4:  1416.662896780108


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [36]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [37]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [38]:
data['persona_fexp'] = 1 * data['fexp']

In [39]:
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data[col_pobres] = (
        (data[col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [40]:
data[['pobres_t1', 'pobres_t2', 'pobres_t3', 'pobres_t4']].sum()

pobres_t1    6188
pobres_t2    7165
pobres_t3    7341
pobres_t4    8514
dtype: int64

In [41]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['ingr_t_t1'] >= 0]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['ingr_t_t2'] >= 0]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t1:  0.22910481442451283
pobreza t2:  0.26169134316983633
pobreza t3:  0.2633870996311457
pobreza t4:  0.3060415127390035


In [42]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [43]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.229105,0.084351,0.044528,NaN,NaN,NaN,NaN
t2,0.261691,0.097322,0.051391,NaN,NaN,NaN,NaN
t3,0.263387,0.100183,0.053562,NaN,NaN,NaN,NaN
t4,0.306042,0.120211,0.064775,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [44]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [45]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.229105,0.084351,0.044528,0.100297,0.188049,0.265789,NaN
t2,0.261691,0.097322,0.051391,0.099763,0.187053,0.26437,NaN
t3,0.263387,0.100183,0.053562,0.099503,0.186747,0.264288,NaN
t4,0.306042,0.120211,0.064775,0.10008,0.187593,0.265033,NaN


Guardamos el ingreso promedio

In [46]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [47]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.229105,0.084351,0.044528,0.100297,0.188049,0.265789,2560.970525
t2,0.261691,0.097322,0.051391,0.099763,0.187053,0.26437,2350.325792
t3,0.263387,0.100183,0.053562,0.099503,0.186747,0.264288,2329.154265
t4,0.306042,0.120211,0.064775,0.10008,0.187593,0.265033,2103.320495


### Inserta los cálculos en la base final

In [49]:
indices = pd.read_csv("indices.csv", encoding='latin-1')

In [51]:
ano = 1991
# Asegurar que el índice de datos_final coincide con trimestres 1..4
datos_final = datos_final.copy()
datos_final["trimestre"] = [1, 2, 3, 4]
datos_final["Año"] = ano

# Reemplazar en indices usando mask
for col in ["fgt0","fgt1","fgt2","a25","a50","a75","ingreso_promedio"]:
    indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values

C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1450961308.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.22910481442451283 0.26169134316983633 0.2633870996311457
 0.3060415127390035]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1450961308.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.0843508437152827 0.09732231973437117 0.10018277695996365
 0.12021130465431924]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
C:\Users\oscarj\AppData\Local\Temp\ipykernel_28312\1450961308.py:9: FutureWarning: Setting an item of incompatible dtype is d

In [53]:
indices.to_csv('indices.csv', encoding='latin-1', index=None)